In [ ]:
!pip install -q langchain langgraph groq gradio requests

In [ ]:
import os
import requests
import gradio as gr
from typing import TypedDict
from langgraph.graph import StateGraph, END
from groq import Groq

In [ ]:
GROQ_API_KEY = "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"
SERPER_API_KEY = "d592c0f7412d92a66d94708a8a3479bbf28383fa"

client = Groq(api_key=GROQ_API_KEY)

In [ ]:
MODEL_NAME = "llama-3.1-8b-instant"

def call_llama(prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

In [ ]:
class BookingState(TypedDict):
    user_input: str
    movie: str
    location: str
    date: str
    theaters: str
    showtime: str
    seats: str
    payment_status: str
    confirmation: str

In [ ]:
def intent_extraction(state: BookingState):

    prompt = f"""
    Extract the following from the request:

    - Movie Name
    - City/Location
    - Date
    - Number of Seats

    User Request:
    {state['user_input']}

    Return format:
    Movie:
    Location:
    Date:
    Seats:
    """

    response = call_llama(prompt)

    for line in response.split("\n"):
        if "Movie:" in line:
            state["movie"] = line.split("Movie:")[1].strip()
        if "Location:" in line:
            state["location"] = line.split("Location:")[1].strip()
        if "Date:" in line:
            state["date"] = line.split("Date:")[1].strip()
        if "Seats:" in line:
            state["seats"] = line.split("Seats:")[1].strip()

    return state

In [ ]:
def search_theaters(state: BookingState):

    query = f"{state['movie']} movie theaters in {state['location']}"

    url = "https://google.serper.dev/search"
    payload = {"q": query}
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.post(url, json=payload, headers=headers)
    results = response.json()

    theaters = []
    for item in results.get("organic", [])[:5]:
        theaters.append(item.get("title", ""))

    state["theaters"] = "\n".join(theaters)

    return state

In [ ]:
def recommend_showtime(state: BookingState):

    prompt = f"""
    Recommend 3 realistic showtimes for:

    Movie: {state['movie']}
    Location: {state['location']}
    Date: {state['date']}

    Provide only times like:
    10:00 AM
    1:30 PM
    6:45 PM
    """

    state["showtime"] = call_llama(prompt)
    return state

In [ ]:
def confirm_booking(state: BookingState):

    state["confirmation"] = f"""
    🎬 Movie: {state['movie']}
    📍 Location: {state['location']}
    📅 Date: {state['date']}
    🎟 Seats: {state['seats']}

    Available Theaters:
    {state['theaters']}

    Suggested Showtimes:
    {state['showtime']}
    """

    return state

In [ ]:
def payment_agent(state: BookingState):

    state["payment_status"] = "Payment Successful ✅"

    state["confirmation"] += f"""

    💳 Payment Status: {state['payment_status']}

    🎉 Booking Confirmed!
    Enjoy your movie.
    """

    return state

In [ ]:
workflow = StateGraph(BookingState)

workflow.add_node("Intent", intent_extraction)
workflow.add_node("Search", search_theaters)
workflow.add_node("Recommend", recommend_showtime)
workflow.add_node("Confirm", confirm_booking)
workflow.add_node("Payment", payment_agent)

workflow.set_entry_point("Intent")

workflow.add_edge("Intent", "Search")
workflow.add_edge("Search", "Recommend")
workflow.add_edge("Recommend", "Confirm")
workflow.add_edge("Confirm", "Payment")
workflow.add_edge("Payment", END)

app = workflow.compile()

In [ ]:
custom_css = """
body {
    background-color: #f4f8fb;
}
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
}
h1 {
    color: #1e3a8a;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🎬 AI Movie Ticket Booking Agent  
    Powered by Llama 3.1 (Groq) + LangGraph + Serper API
    """)

    with gr.Tab("Search & Book"):

        user_input = gr.Textbox(
            label="Enter Booking Request",
            placeholder="Book 2 tickets for Dune in Singapore tomorrow evening",
            lines=4
        )

        book_button = gr.Button("Search & Book Tickets", variant="primary")

        output = gr.Markdown()

        def run_booking(request):

            state = {
                "user_input": request,
                "movie": "",
                "location": "",
                "date": "",
                "theaters": "",
                "showtime": "",
                "seats": "",
                "payment_status": "",
                "confirmation": ""
            }

            result = app.invoke(state)
            return result["confirmation"]

        book_button.click(
            run_booking,
            inputs=user_input,
            outputs=output
        )

    with gr.Tab("Payment Info"):
        gr.Markdown("💳 Secure payment simulation enabled.")
        gr.Markdown("All transactions are securely processed (demo mode).")

demo.launch(debug=True)